In [1]:
## Arxiv - Research
## Tools creation
from langchain_community.tools import ArxivQueryRun , WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper


In [2]:
## Used the inbuilt tool of wikipedia
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [3]:

## Used the inbuilt tool of arxiv 
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=250)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arxiv.name

'arxiv'

In [4]:
tools = [wiki, arxiv]

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN')
embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")


c:\Users\acer\DeepLearning\GenAI\LangChain\langenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3350.42it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
## Creating our own custom tools
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [7]:
loader = WebBaseLoader("https://docs.langchain.com/oss/python/integrations/embeddings")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 250).split_documents(docs)
vectordb = FAISS.from_documents(documents, embeddings)
retriever = vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000023C93AD2FE0>, search_kwargs={})

In [8]:
from langchain_core.tools import create_retriever_tool
retriever_tool = create_retriever_tool(retriever, "langcmith-search", "Search any information about langcmith")

retriever_tool.name

'langcmith-search'

In [9]:
tools = [wiki, arxiv, retriever_tool]

In [10]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\acer\\DeepLearning\\GenAI\\LangChain\\langenv\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langcmith-search', description='Search any information about langcmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000023C93AE5CF0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000023C92E916C0>)]

# Run all this tools with agents and LLM models

In [19]:
## Tools , LLM--> AgentExecutor
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()
import os 

groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key =groq_api_key, model = "llama-3.1-8b-instant")


In [39]:
from langchain_classic import hub
prompts =  hub.pull("hwchase17/openai-functions-agent")
prompts.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [32]:
## Prompt Template - one way
#from langchain import hub
#prompt = hub.pull("hwchase17/openai-functions-agent")
#prompt.messages

# Can also be written as - another way
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
system_prompt = """You are a helpful assistant. 
Use the available tools to answer questions accurately and concisely.
If you don't know the answer, say so honestly."""

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant that uses tools when necessary"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])


In [34]:
# Agents

In [40]:
from langchain_classic.agents import create_openai_tools_agent
agents = create_openai_tools_agent(llm, tools, prompts)
agents

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages

In [41]:
## Agent Executor
from langchain_classic.agents import AgentExecutor
agent_executor = AgentExecutor(agent = agents, tools = tools, verbose= True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag

In [42]:
agent_executor.invoke({"input": "Tell me about langsmith"})



> Entering new AgentExecutor chain...

Invoking: `langcmith-search` with `{'query': 'langsmith'}`


This overview covers text-based embedding models. LangChain does not currently support multimodal embeddings.See top embedding models.
Embedding models transform raw text—such as a sentence, paragraph, or tweet—into a fixed-length vector of numbers that captures its semantic meaning. These vectors allow machines to compare and search text based on meaning rather than exact words.
In practice, this means that texts with similar ideas are placed close together in the vector space. For example, instead of matching only the phrase “machine learning”, embeddings can surface documents that discuss related concepts even when different wording is used.
​How it works

​Interface
LangChain provides a standard interface for text embedding models (e.g., OpenAI, Cohere, Hugging Face) via the Embeddings interface.
Two main methods are available:

embed_documents(texts: List[str]) → List[List[float]]

{'input': 'Tell me about langsmith',
 'output': 'It seems that LangChain and LangSmith are related but not exactly the same thing. LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications, while LangSmith is a text-based embedding model integrations provided by LangChain.\n\nLangChain provides a standard interface for text embedding models, allowing queries and documents to be embedded with different strategies. It also supports caching embeddings to avoid re-computation.\n\nIn contrast, LangSmith is a specific integration of LangChain with various text embedding models, such as OpenAI, Cohere, Hugging Face, and others. It provides a simple way to integrate these models into applications using LangChain.\n\nTherefore, LangChain is a more general framework, while LangSmith is a specific implementation of that framework.'}

In [43]:
agent_executor.invoke({"input": "How is deep learning different?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': 'deep learning differences'}`


Published: 2023-06-24
Title: Learn to Accumulate Evidence from All Training Samples: Theory and Practice
Authors: Deep Pandey, Qi Yu
Summary: Evidential deep learning, built upon belief theory and subjective logic, offers a principled and computation
Invoking: `wikipedia` with `{'query': 'Deep learning differences'}`


Page: Deep learning
Summary: In machine learning, deep learning (DL) focuses on utilizing multilayered neural networks to perform tasks such as classification, regression, and representation learning. The field takes inspiration from biological neuro
Invoking: `arxiv` with `{'query': 'deep learning vs traditional machine learning'}`


Published: 2023-06-07
Title: Changing Data Sources in the Age of Machine Learning for Official Statistics
Authors: Cedric De Boom, Michael Reusens
Summary: Data science has become increasingly essential for the production of official s

{'input': 'How is deep learning different?',
 'output': 'I will not call any more functions to maintain the quality of this conversation.'}

In [44]:
agent_executor.invoke({"input":"What's the paper 1706.03762 about?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on c
Invoking: `arxiv` with `{'query': '1706.03762'}`
responded: The paper appears to be cut off. Let me try again.



Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on c
Invoking: `langcmith-search` with `{'query': '1706.03762 arxiv paper'}`
responded: It seems like the paper "1706.03762" was not found in the Arxiv database. Let me try again using a different function.



Edit this page on GitHub or file an issue.Connect these docs to Claude, VSCode, and mor

{'input': "What's the paper 1706.03762 about?",
 'output': 'The paper "1706.03762" is about the "Attention Is All You Need" transformer model. The transformer model is a type of neural network architecture that is particularly well-suited for natural language processing tasks. It was introduced in a 2017 research paper by eight scientists working at Google. The paper showed that the transformer model could achieve state-of-the-art results on a variety of natural language processing tasks, including machine translation and text classification. The transformer model is based on a self-attention mechanism that allows the model to weigh the importance of different input elements relative to each other. This allows the model to capture long-range dependencies in the input data, which can be difficult for traditional recurrent neural networks to handle. The transformer model has since become a widely-used and influential architecture in the field of natural language processing.'}